## Tahap 1: Data Understanding

In [33]:
import pandas as pd

df = pd.read_csv("rawdata.csv")
df["Id"] = range(1, len(df) + 1)
df = df[["Id", "Nama", "Rating", "Jumlah Ulasan", "Website", "NoTelp", "Alamat", "Status"]]

print("Jumlah data awal:", len(df))
print()
df.info()

Jumlah data awal: 145

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Id             145 non-null    int64 
 1   Nama           145 non-null    object
 2   Rating         145 non-null    object
 3   Jumlah Ulasan  145 non-null    int64 
 4   Website        38 non-null     object
 5   NoTelp         118 non-null    object
 6   Alamat         141 non-null    object
 7   Status         145 non-null    object
dtypes: int64(2), object(6)
memory usage: 9.2+ KB


In [34]:
print("Distribusi label:")
print(df["Status"].value_counts())
print()
print("Jumlah nilai kosong per kolom:")
print(df.isnull().sum())

Distribusi label:
Status
Cold Prospek    83
Hot Prospek     62
Name: count, dtype: int64

Jumlah nilai kosong per kolom:
Id                 0
Nama               0
Rating             0
Jumlah Ulasan      0
Website          107
NoTelp            27
Alamat             4
Status             0
dtype: int64


## Tahap 2: Data Preparation

In [35]:
df = df.dropna(subset=["NoTelp"])
print("Jumlah data setelah drop NoTelp kosong:", len(df))

Jumlah data setelah drop NoTelp kosong: 118


In [36]:
df_model = df[["Id", "Rating", "Jumlah Ulasan", "Website", "Status"]].copy()

df_model["Rating"] = df_model["Rating"].astype(str).str.replace(",", ".", regex=False)
df_model["Jumlah Ulasan"] = df_model["Jumlah Ulasan"].astype(str).str.replace(".", "", regex=False)
df_model["Website"] = df_model["Website"].fillna("").apply(lambda x: 1 if x != "" else 0)
df_model["Status"] = df_model["Status"].apply(lambda x: 1 if x != "Cold Prospek" else 0)

df_model["Rating"] = pd.to_numeric(df_model["Rating"], errors="coerce")
df_model["Jumlah Ulasan"] = pd.to_numeric(df_model["Jumlah Ulasan"], errors="coerce")
df_model = df_model.dropna(subset=["Rating", "Jumlah Ulasan"])

print("Jumlah data siap modeling:", len(df_model))
print()
print("Distribusi label (0 = Cold Prospek, 1 = Hot Prospek):")
print(df_model["Status"].value_counts())
print()
df_model.head()

Jumlah data siap modeling: 118

Distribusi label (0 = Cold Prospek, 1 = Hot Prospek):
Status
1    62
0    56
Name: count, dtype: int64



,Id,Rating,Jumlah Ulasan,Website,Status
0,1,4.7,127,1,0
1,2,4.5,250,0,0
2,3,4.5,2200,0,0
3,4,4.9,29,0,1
5,6,4.6,426,0,0


In [37]:
from sklearn.model_selection import train_test_split

X = df_model[["Rating", "Jumlah Ulasan", "Website"]]
y = df_model["Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Jumlah data keseluruhan :", len(df_model))
print("Data training          :", len(X_train))
print("Data testing           :", len(X_test))

Jumlah data keseluruhan : 118
Data training          : 94
Data testing           : 24


## Tahap 3: Modeling

In [38]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    max_depth=None,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Model selesai dilatih.")
print("Fitur yang digunakan:", rf_model.feature_names_in_.tolist())

Model selesai dilatih.
Fitur yang digunakan: ['Rating', 'Jumlah Ulasan', 'Website']


In [39]:
feature_importance = pd.DataFrame({
    "Fitur"     : X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("Feature Importance:")
print(feature_importance)

Feature Importance:
           Fitur  Importance
0  Jumlah Ulasan    0.688556
1         Rating    0.283018
2        Website    0.028426


## Tahap 4: Evaluasi Model

### 4.1 Tanpa SMOTE

In [40]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print()
print(f"True Negative  (TN) : {tn}  — Cold diprediksi Cold")
print(f"False Positive (FP) : {fp}  — Cold diprediksi Hot")
print(f"False Negative (FN) : {fn}  — Hot diprediksi Cold")
print(f"True Positive  (TP) : {tp}  — Hot diprediksi Hot")

Confusion Matrix:
[[ 4  7]
 [ 2 11]]

True Negative  (TN) : 4  — Cold diprediksi Cold
False Positive (FP) : 7  — Cold diprediksi Hot
False Negative (FN) : 2  — Hot diprediksi Cold
True Positive  (TP) : 11  — Hot diprediksi Hot


In [41]:
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print("Hasil Evaluasi (Tanpa SMOTE):")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")

Hasil Evaluasi (Tanpa SMOTE):
  Accuracy  : 0.6250
  Precision : 0.6111
  Recall    : 0.8462
  F1-Score  : 0.7097
  AUC-ROC   : 0.6469


### 4.2 Dengan SMOTE

In [42]:
from imblearn.over_sampling import SMOTE

X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)

print("Distribusi label setelah SMOTE (data training):")
print(pd.Series(y_train_sm).value_counts())

Distribusi label setelah SMOTE (data training):
Status
0    49
1    49
Name: count, dtype: int64


/home/enjuy/Documents/Ngoding/Python/Python---Random-Forest/venv/lib/python3.12/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [43]:
rf_smote = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    random_state=42
)

rf_smote.fit(X_train_sm, y_train_sm)
y_pred_sm = rf_smote.predict(X_test)

cm_sm = confusion_matrix(y_test, y_pred_sm)
tn2, fp2, fn2, tp2 = cm_sm.ravel()

print("Confusion Matrix (SMOTE):")
print(cm_sm)
print()
print(f"True Negative  (TN) : {tn2}")
print(f"False Positive (FP) : {fp2}")
print(f"False Negative (FN) : {fn2}")
print(f"True Positive  (TP) : {tp2}")

Confusion Matrix (SMOTE):
[[ 5  6]
 [ 2 11]]

True Negative  (TN) : 5
False Positive (FP) : 6
False Negative (FN) : 2
True Positive  (TP) : 11


In [44]:
accuracy_sm  = accuracy_score(y_test, y_pred_sm)
precision_sm = precision_score(y_test, y_pred_sm)
recall_sm    = recall_score(y_test, y_pred_sm)
f1_sm        = f1_score(y_test, y_pred_sm)
auc_sm       = roc_auc_score(y_test, rf_smote.predict_proba(X_test)[:, 1])

print("Hasil Evaluasi (Dengan SMOTE):")
print(f"  Accuracy  : {accuracy_sm:.4f}")
print(f"  Precision : {precision_sm:.4f}")
print(f"  Recall    : {recall_sm:.4f}")
print(f"  F1-Score  : {f1_sm:.4f}")
print(f"  AUC-ROC   : {auc_sm:.4f}")

Hasil Evaluasi (Dengan SMOTE):
  Accuracy  : 0.6667
  Precision : 0.6471
  Recall    : 0.8462
  F1-Score  : 0.7333
  AUC-ROC   : 0.6643


### 4.3 Cross-Validation (5-Fold)

In [45]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracy = cross_val_score(rf_model, X, y, cv=cv, scoring="accuracy")
cv_recall   = cross_val_score(rf_model, X, y, cv=cv, scoring="recall")
cv_f1       = cross_val_score(rf_model, X, y, cv=cv, scoring="f1")

print("Hasil Cross-Validation 5-Fold:")
print(f"  Accuracy : {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
print(f"  Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")
print(f"  F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")

Hasil Cross-Validation 5-Fold:
  Accuracy : 0.6609 (+/- 0.1068)
  Recall   : 0.6603 (+/- 0.1352)
  F1-Score : 0.6693 (+/- 0.1033)


### 4.4 Perbandingan Hasil Evaluasi

In [46]:
hasil = pd.DataFrame({
    "Metrik"       : ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"],
    "Tanpa SMOTE"  : [accuracy, precision, recall, f1, auc],
    "Dengan SMOTE" : [accuracy_sm, precision_sm, recall_sm, f1_sm, auc_sm],
})

hasil[["Tanpa SMOTE", "Dengan SMOTE"]] = hasil[["Tanpa SMOTE", "Dengan SMOTE"]].map(lambda x: round(x, 4))
print(hasil.to_string(index=False))

   Metrik  Tanpa SMOTE  Dengan SMOTE
 Accuracy       0.6250        0.6667
Precision       0.6111        0.6471
   Recall       0.8462        0.8462
 F1-Score       0.7097        0.7333
  AUC-ROC       0.6469        0.6643


## Tahap 5: Simpan Model

In [47]:
import pickle, os

MODEL_PATH = os.path.join(os.getcwd(), "model.pkl")

with open(MODEL_PATH, "wb") as f:
    pickle.dump(rf_model, f)

print(f"Model berhasil disimpan ke : {MODEL_PATH}")
print(f"Ukuran file                : {os.path.getsize(MODEL_PATH):,} bytes")

Model berhasil disimpan ke : /home/enjuy/Documents/Ngoding/Python/Python---Random-Forest/EDA/model.pkl
Ukuran file                : 438,318 bytes


In [48]:
with open(MODEL_PATH, "rb") as f:
    loaded_model = pickle.load(f)

y_pred_verify = loaded_model.predict(X_test)
match = (y_pred_verify == y_pred).all()

print(f"Verifikasi prediksi identik : {match}")
print(f"Jumlah estimator            : {loaded_model.n_estimators}")
print(f"Fitur yang digunakan        : {loaded_model.feature_names_in_.tolist()}")

Verifikasi prediksi identik : True
Jumlah estimator            : 100
Fitur yang digunakan        : ['Rating', 'Jumlah Ulasan', 'Website']
